# K-Strategy Panel — Viewer

Analisi interattiva dei risultati prodotti da `iq k-analyze`.

**Non calcola nulla** — legge i file già prodotti dalla CLI:
- `outputs/WFO_T_DEV_RESULTS/classification_*.csv`
- `outputs/WFO_T_DEV_RESULTS/<strategy>/*_results.pkl`

**Flusso:**
```
iq k-analyze --ptf <nome> → classification.csv + _results.pkl
questo JN → carica, filtra, visualizza, esporta
```


In [ ]:
%run _bootstrap_dev.ipynb

## §1 — Configurazione


In [ ]:
from pathlib import Path
from datetime import datetime
from IPython.display import display
import plotly.express as px
WFO_DIR     = Path(_TSLAB_DEV_T_WFO_RESULTS_DIR)
EXPORT_DIR  = Path(_TSLAB_K_PANEL_EXPORTS_DIR)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


CSV_PATTERN     = 'classification_*.csv'
FILTER_PROMOTED = 'ALL'       # 'ALL' | 'PROMOTED' | 'FAILED'
SORT_BY         = 'Sharpe'    # 'Sharpe' | 'CAGR%' | 'TotRet%' | 'DSR' | 'MaxDD%'
SORT_ASC        = False
RATIO           = '4:1'

print(f'WFO dir  : {WFO_DIR}')
print(f'Export   : {EXPORT_DIR}')
print(f'Filtro   : {FILTER_PROMOTED} | Sort: {SORT_BY} asc={SORT_ASC}')


## §2 — Classifica


In [ ]:
df = load_k_classifications(WFO_DIR, CSV_PATTERN, FILTER_PROMOTED, SORT_BY, SORT_ASC)

print(f'Totale  : {len(df)}')
print(f'Promossi: {(df["Promoted"]=="PASS").sum()}')
print(f'Falliti : {(df["Promoted"]=="FAIL").sum()}')
print()
display(style_k_classification(df))

## §3 — Scatter Sharpe vs TotRet%


In [ ]:
fig = px.scatter(
    df, x='Sharpe', y='TotRet%',
    color='Promoted',
    color_discrete_map={'PASS': '#28a745', 'FAIL': '#dc3545'},
    hover_name=df['Ticker'] + '@' + df['Strategy'],
    hover_data={
        'Sharpe': ':.3f',
        'TotRet%': ':.1f',
        'BH_TotRet%': ':.1f',
        'DeltaTotRet%': ':.1f',
        'MaxDD%': ':.1f',
        'BH_DD%': ':.1f',
        'DeltaDD%': ':.1f',
        'DSR': ':.3f',
        'OFC': True,
        'MC': True,
        'Promoted': True,
    },
    title='Sharpe vs TotRet% — tutti i trading system',
    labels={'Sharpe': 'Sharpe Ratio', 'TotRet%': 'Total Return %'},
    template='plotly_white',
)
fig.update_traces(
    marker_size=10,
    mode='markers+text',
    text=df['Ticker'] + '@' + df['Strategy'],
    textposition='top right',
    textfont=dict(size=14),
)
fig.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.4)
fig.add_vline(x=0, line_dash='dash', line_color='gray', opacity=0.4)
fig.update_layout(height=560)
fig.show()


## §4 — Equity curve promossi


In [ ]:
promoted = df[df['Promoted'] == 'PASS']
print(f'Promossi: {len(promoted)}')

for _, row in promoted.iterrows():
    ticker, strategy = row['Ticker'], row['Strategy']
    pf, bh = load_k_equity(WFO_DIR, ticker, strategy, RATIO)
    if pf is None:
        print(f'  {ticker}@{strategy}: pkl non trovato')
        continue
    plot_k_equity(ticker, strategy, pf, bh, row).show()


## §5 — Confronto run


In [ ]:
df_all_runs = load_k_classifications(WFO_DIR, CSV_PATTERN, 'ALL', 'Sharpe', False)

if df_all_runs['_run'].nunique() > 1:
    print(f'Run disponibili: {sorted(df_all_runs["_run"].unique())}')
    pivot = df_all_runs.pivot_table(
        index=['Ticker', 'Strategy'], columns='_run',
        values='Sharpe', aggfunc='first'
    ).round(3)
    display(pivot)
else:
    print('Un solo run disponibile.')
    print('Lancia iq k-analyze piu volte per confrontare run diversi.')


## §6 — Export promossi


In [ ]:
promoted = df[df['Promoted'] == 'PASS'].copy()
ts = datetime.now().strftime('%Y%m%d_%H%M%S')

# 1. CSV completo promossi
csv_path = EXPORT_DIR / f'promoted_{ts}.csv'
promoted.drop(columns=['_run'], errors='ignore').to_csv(csv_path, index=False)
print(f'CSV salvato: {csv_path}')

# 2. Snippet trading_systems per k_portfolios.py
py_path = EXPORT_DIR / f'trading_systems_{ts}.py'
lines = [
    f'# Generato da k_strategy_panel — {ts}',
    f'# Promossi: {len(promoted)} trading system',
    '',
    'trading_systems = [',
]
for _, row in promoted.iterrows():
    lines.append(
        f'    {{"symbol": "{BOLD}{row["Ticker"]}{RESET}", "strategy": "{BOLD}{row["Strategy"]}{RESET}"}},'
        f'  # Sharpe={row["Sharpe"]:.3f}'
        f' TotRet={row["TotRet%"]:.1f}% (BH={row["BH_TotRet%"]:.1f}%)'
        f' MaxDD={row["MaxDD%"]:.1f}% (BH={row["BH_DD%"]:.1f}%)'
    )
lines.append(']')
py_path.write_text('\n'.join(lines))
print(f'Snippet salvato: {py_path}')
print()
print('\n'.join(lines))


## § 7 Trading System Analysis
Carica un trading system dal disco e analizzalo in profondità.

### Promoted — Analisi Comparativa per Ticker

In [ ]:
# Carica classifications e lancia analisi comparativa su tutti i promoted
_df_class = load_k_classifications(WFO_DIR)
_ = analyze_promoted_ts(
    classification_df=_df_class,
    wfo_dir=WFO_DIR,
    # ratio=ratio,
)

In [ ]:
# Analisi di un singolo TS

strategy = 'bollinger'   # nome strategia
symbol   = 'AMD'             # ticker

# ratio  = '4:1'             # decommentare se necessario
portfolio, bh_portfolio, wfo_results = load_ts(
    symbol=symbol,
    strategy=strategy,
    # ratio=ratio,
    wfo_results_dir=WFO_DIR,
)

### Exposure Regime

In [ ]:
df_exposure, df_exposure_year = analyze_exposure_regime(
    pf=portfolio,
    title=f"{symbol} {strategy} - Exposure Regime",
    show_report=True,
)

### Timing Efficiency

In [ ]:
out = analyze_timing_efficiency(
    pf=portfolio,
    portfolio_type="auto",
    show_report=True,
    vbt_plot_width=1100,
)

### Portfolio Performance

In [ ]:
auto_adjust = True # tipicamente nel calcolo performance uso prezi rettificati per ottenere Total Return. Se False -> Price return

portfolio_title = f"{symbol} - Total"
portfolio_title += " Return" if auto_adjust else " Price" 


figs = generate_portfolio_performance(
    pf=portfolio,
    portfolio_title=portfolio_title,
    benchmark="Internal Benchmark (B&H)",
    alpha_analysis=True,
    show_plots=True
)